# ARC-v0.31 — Frozen Generative Retrieval-Agent Transfer

**Goal.** Test whether the ANN mechanism boundary observed under centroid-style feedback transfers to a **frozen generative query-rewrite agent**.

This notebook deliberately keeps the first agent experiment narrow:

- **Dataset:** BEIR Natural Questions, reusing the NQ-GTE validation pool.
- **Representation contrast:** IVF-PQ32@64 → IVF-SQ8@64.
- **Search-effort contrast:** IVF-SQ8 nprobe 2 → 64.
- **Agent operator:** one frozen LLM that rewrites the next retrieval query from the original question, current query, and retrieved evidence.
- **No scores in the prompt.**
- **No adaptive stopping.**
- **H=4 updates.**
- **32-query engineering smoke set + 500-query frozen main set.**
- **Primary:** representation-minus-search `H3abs` under the generative operator, with a two-sided paired-query bootstrap interpretation.

This is prospective for the **new agent outcomes**, but is not a pristine new-data confirmation because the NQ validation pool was already used in ARC-v0.27/v0.28.

Protocol template SHA-256:

`67108a1f3ddb36776339555e0a09d8fb65dac18ab5a9979990760b27706b9482`

### Scientific retention rule

A positive transfer, null, reversal, or qualitatively different result is valid. Do **not** retune the agent prompt, model, ANN comparators, horizon, endpoint, or main-query subset after main outcomes begin.

In [ ]:
# Cell 1 — Install/import, mount Drive, immutable configuration
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass
import gc, hashlib, json, math, os, random, re, shutil, time, warnings

import numpy as np
import pandas as pd

try:
    import faiss
except Exception:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "faiss-cpu", "pyarrow", "sentence-transformers",
                           "transformers", "accelerate", "ir_datasets",
                           "huggingface_hub", "tqdm"])
    import faiss

try:
    import torch
    from sentence_transformers import SentenceTransformer
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from huggingface_hub import model_info
    import ir_datasets
    from tqdm.auto import tqdm
except Exception:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "sentence-transformers", "transformers", "accelerate",
                           "ir_datasets", "huggingface_hub", "tqdm"])
    import torch
    from sentence_transformers import SentenceTransformer
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from huggingface_hub import model_info
    import ir_datasets
    from tqdm.auto import tqdm

try:
    from google.colab import drive
except ImportError:
    drive = None

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 20260827
np.random.seed(SEED)
random.seed(SEED)

# ----- Frozen experiment knobs -----
DATASET_ID = "beir/nq"
DIM = 384
N_DOCS_EXPECTED = 2_681_468

H = 4
SEARCH_K = 100
UTILITY_K = 10
AGENT_EVIDENCE_K = 5

N_SMOKE = 32
N_MAIN = 500
BOOTSTRAP_REPS = 10_000

HIGH_NPROBE = 64
MATCHED_LOW_NPROBE = 2

AGENT_BACKEND = "transformers_local"
AGENT_MODEL = "Qwen/Qwen2.5-3B-Instruct"
AGENT_MAX_NEW_TOKENS = 48
AGENT_MAX_PASSAGE_CHARS = 850

ENCODER_MODEL = "thenlper/gte-small"

DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.is_dir():
    if drive is None:
        raise RuntimeError("Google Drive unavailable.")
    drive.mount("/content/drive")

RAG_ROOT = DRIVE_ROOT / "rag-pq-checkpoints"
ARC_ROOT = RAG_ROOT / "arc-v0"
assert RAG_ROOT.is_dir(), RAG_ROOT
assert ARC_ROOT.is_dir(), ARC_ROOT

OUT_ROOT = ARC_ROOT / "generative-retrieval-agent-transfer-v031"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

print("FAISS:", getattr(faiss, "__version__", "unknown"))
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("ARC root:", ARC_ROOT)
print("Output root:", OUT_ROOT)

## Experimental principle

The agent is a **state-transition operator**, not a new retrieval method.  
For each mechanism, lower- and higher-fidelity branches use the **same** agent model and prompt. Only the ANN intervention differs.

At round \(t\):

\[
D_t^r = R_r(q_t^r), \qquad
q_{t+1}^r = A(Q, q_t^r, D_t^r)
\]

where \(A\) is the frozen generative query-rewrite agent.

The prompt receives document **rank/title/text**, but not ANN scores. This prevents the representation score channel from being silently mixed into the first agent transfer experiment.

In [ ]:
# Cell 2 — Locate frozen v0.28/v0.27 artifacts without guessing one exact filename

V028_ROOT = ARC_ROOT / "nq-gte-operator-horizon-generalization-v028"
V027_CACHE = RAG_ROOT / "arc-v027-nq-gte-large-cache"

def existing_roots():
    roots = []
    for p in [V028_ROOT, V027_CACHE]:
        if p.exists():
            roots.append(p)
    for p in ARC_ROOT.iterdir():
        n = p.name.lower()
        if ("nq" in n and ("gte" in n or "v027" in n or "v028" in n)) and p not in roots:
            roots.append(p)
    return roots

ROOTS = existing_roots()
print("Candidate NQ roots:")
for p in ROOTS:
    print(" -", p)

# v0.28 endpoint parquet gives the authoritative 1,726-query validation membership.
endpoint_candidates = []
for root in ROOTS:
    endpoint_candidates.extend(root.rglob("*v028*h50*endpoint*.parquet"))
    endpoint_candidates.extend(root.rglob("*h50*endpoint*.parquet"))
endpoint_candidates = sorted(set(endpoint_candidates), key=lambda p: (len(str(p)), str(p)))

if not endpoint_candidates:
    raise FileNotFoundError(
        "Could not locate the ARC-v0.28 H50 endpoint parquet. "
        "Set V028_ENDPOINT manually after inspecting the Drive paths."
    )

# Prefer the official run used by the paper if present.
official = [p for p in endpoint_candidates if "20260825-181541" in str(p)]
V028_ENDPOINT = official[0] if official else endpoint_candidates[0]
print("Using v0.28 endpoint:", V028_ENDPOINT)

e28 = pd.read_parquet(V028_ENDPOINT, columns=["query_id"])
VAL_IDS = sorted(e28["query_id"].astype(str).unique().tolist())
assert len(VAL_IDS) == 1726, len(VAL_IDS)

def hash_order(qid):
    return hashlib.sha256(f"{SEED}|{qid}".encode()).hexdigest()

ordered = sorted(VAL_IDS, key=hash_order)
SMOKE_IDS = ordered[:N_SMOKE]
MAIN_IDS = ordered[N_SMOKE:N_SMOKE+N_MAIN]
assert set(SMOKE_IDS).isdisjoint(MAIN_IDS)
assert len(MAIN_IDS) == N_MAIN

sample_manifest = {
    "seed": SEED,
    "source_endpoint": str(V028_ENDPOINT),
    "n_pool": len(VAL_IDS),
    "smoke_ids": SMOKE_IDS,
    "main_ids": MAIN_IDS,
    "selection": "SHA256(seed|qid) lexicographic order"
}
sample_text = json.dumps(sample_manifest, indent=2, sort_keys=True) + "\n"
sample_sha = hashlib.sha256(sample_text.encode()).hexdigest()

print("Validation pool:", len(VAL_IDS))
print("Smoke:", len(SMOKE_IDS), "Main:", len(MAIN_IDS))
print("Sample manifest SHA:", sample_sha)

In [ ]:
# Cell 3 — Discover the NQ PQ32/SQ8 indexes and classify them by FAISS structure
# Optional manual overrides. Leave as None for auto-discovery.
PQ_PATH_OVERRIDE = None
SQ_PATH_OVERRIDE = None

# Known authoritative persisted v0.27 index paths.
KNOWN_PQ_PATH = V027_CACHE / "nq-gte-ivfpq-nlist4096-m32-nbits8.faiss"
KNOWN_SQ_PATH = V027_CACHE / "nq-gte-ivfsq8-nlist4096.faiss"


def candidate_index_files():
    out = []
    for root in ROOTS:
        for pattern in ("*.faiss", "*.index"):
            out.extend(root.rglob(pattern))
    # Prefer filenames with nq/gte/pq/sq markers.
    return sorted(set(out), key=lambda p: (
        0 if any(k in p.name.lower() for k in ["nq", "gte", "pq", "sq"]) else 1,
        len(str(p)), str(p)
    ))

idx_files = candidate_index_files()
print("Index candidates:", len(idx_files))
for p in idx_files[:30]:
    print(" -", p)

def load_and_describe(path):
    idx = faiss.read_index(str(path))
    return idx, type(idx).__name__, int(idx.ntotal), int(idx.d)

if PQ_PATH_OVERRIDE:
    PQ_PATH = Path(PQ_PATH_OVERRIDE)
elif KNOWN_PQ_PATH.is_file():
    PQ_PATH = KNOWN_PQ_PATH
else:
    PQ_PATH = next((
        p for p in idx_files
        if ("pq32" in p.name.lower() or "m32" in p.name.lower())
        and "sq" not in p.name.lower()
    ), None)

if SQ_PATH_OVERRIDE:
    SQ_PATH = Path(SQ_PATH_OVERRIDE)
elif KNOWN_SQ_PATH.is_file():
    SQ_PATH = KNOWN_SQ_PATH
else:
    SQ_PATH = next((
        p for p in idx_files
        if "sq8" in p.name.lower() or "scalar" in p.name.lower()
    ), None)

# If filenames are opaque, inspect a few candidates by class.
if PQ_PATH is None or SQ_PATH is None:
    for p in idx_files[:20]:
        try:
            idx, cls, nt, d = load_and_describe(p)
            print(p.name, cls, nt, d)
            if nt == N_DOCS_EXPECTED and d == DIM:
                if PQ_PATH is None and "PQ" in cls.upper():
                    pq_m = getattr(getattr(idx, "pq", None), "M", None)
                    if pq_m in (None, 32):
                        PQ_PATH = p
                if SQ_PATH is None and "SCALAR" in cls.upper():
                    SQ_PATH = p
            del idx
        except Exception as exc:
            print("skip unreadable:", p, repr(exc))

if PQ_PATH is None or SQ_PATH is None:
    raise FileNotFoundError(
        "Could not unambiguously locate the NQ-GTE PQ32 and SQ8 indexes. "
        "Set PQ_PATH_OVERRIDE and SQ_PATH_OVERRIDE in this cell."
    )

print("PQ:", PQ_PATH)
print("SQ:", SQ_PATH)

pq_cpu = faiss.read_index(str(PQ_PATH))
sq_cpu = faiss.read_index(str(SQ_PATH))
assert pq_cpu.ntotal == N_DOCS_EXPECTED and sq_cpu.ntotal == N_DOCS_EXPECTED
assert pq_cpu.d == DIM and sq_cpu.d == DIM

pq_cpu.nprobe = HIGH_NPROBE
sq_cpu.nprobe = HIGH_NPROBE
faiss.omp_set_num_threads(max(1, os.cpu_count() or 1))

print("PQ class:", type(pq_cpu).__name__)
print("SQ class:", type(sq_cpu).__name__)
print("INDEX DISCOVERY — PASS")

## Persisted v0.27 artifacts recovered

"
            "The original NQ-GTE cache is stored directly under `rag-pq-checkpoints/`, not under `rag-pq-checkpoints/arc-v0/`. "
            "The persisted index binaries are therefore reused directly:

"
            "- `arc-v027-nq-gte-large-cache/nq-gte-ivfpq-nlist4096-m32-nbits8.faiss`
"
            "- `arc-v027-nq-gte-large-cache/nq-gte-ivfsq8-nlist4096.faiss`

"
            "This means ARC-v0.31 can retain the original v0.27 ANN artifacts instead of rebuilding a new index ladder.

## Why the corpus-row audit is mandatory

FAISS returns integer row IDs. The agent needs the corresponding passage text.  
We therefore reconstruct BEIR NQ's row-to-document mapping and **abort** if it does not produce sensible retrieval effectiveness. This prevents a silent row-order mismatch from turning the agent experiment into nonsense.

In [ ]:
# Cell 4 — Load BEIR NQ query text/qrels and construct row->doc_id mapping
dataset = ir_datasets.load(DATASET_ID)

QUERY_TEXT = {str(q.query_id): q.text for q in dataset.queries_iter()}
missing_q = [q for q in (SMOKE_IDS + MAIN_IDS) if q not in QUERY_TEXT]
assert not missing_q, f"Missing query text for {len(missing_q)} selected qids"

QRELS_DOC = defaultdict(set)
for qr in dataset.qrels_iter():
    if int(qr.relevance) > 0:
        QRELS_DOC[str(qr.query_id)].add(str(qr.doc_id))

ROW_DOCID_CACHE = OUT_ROOT / "nq_ir_datasets_row_docids.npy"
if ROW_DOCID_CACHE.exists():
    ROW_DOCIDS = np.load(ROW_DOCID_CACHE, allow_pickle=True)
else:
    print("Building one-time BEIR NQ row->doc_id cache (2.68M IDs)...")
    ROW_DOCIDS = np.asarray([str(d.doc_id) for d in tqdm(dataset.docs_iter(), total=N_DOCS_EXPECTED)], dtype=object)
    assert len(ROW_DOCIDS) == N_DOCS_EXPECTED
    np.save(ROW_DOCID_CACHE, ROW_DOCIDS, allow_pickle=True)

assert len(ROW_DOCIDS) == N_DOCS_EXPECTED
DOC_STORE = dataset.docs_store()

print("Queries:", len(QUERY_TEXT))
print("Row/doc IDs:", len(ROW_DOCIDS))
print("Selected qrels coverage:",
      np.mean([len(QRELS_DOC[q]) > 0 for q in MAIN_IDS]))

In [ ]:
# Cell 5 — Load/freeze GTE query encoder and test row semantics on original queries
encoder = SentenceTransformer(ENCODER_MODEL, device="cuda" if torch.cuda.is_available() else "cpu")

def encode_texts(texts, batch_size=128):
    x = encoder.encode(
        list(texts),
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    return np.ascontiguousarray(x, dtype=np.float32)

def search(index, q, nprobe, k=SEARCH_K):
    index.nprobe = int(nprobe)
    s, i = index.search(np.ascontiguousarray(q, dtype=np.float32), int(k))
    return s, i

DISCOUNTS = 1.0 / np.log2(np.arange(2, UTILITY_K + 2))

def metrics_from_rows(qids, rows):
    ndcg, mrr, recall = [], [], []
    for qid, rr in zip(qids, rows):
        docs = [str(ROW_DOCIDS[int(r)]) for r in rr[:UTILITY_K] if int(r) >= 0]
        rel = QRELS_DOC[str(qid)]
        hits = np.asarray([d in rel for d in docs], dtype=np.float64)
        dcg = float((hits * DISCOUNTS[:len(hits)]).sum())
        ideal_n = min(len(rel), UTILITY_K)
        idcg = float(DISCOUNTS[:ideal_n].sum()) if ideal_n else 0.0
        ndcg.append(dcg / idcg if idcg else 0.0)
        hit_positions = np.where(hits > 0)[0]
        mrr.append(1.0 / (hit_positions[0] + 1) if len(hit_positions) else 0.0)
        recall.append(float(hits.sum()) / max(len(rel), 1))
    return np.asarray(ndcg), np.asarray(mrr), np.asarray(recall)

AUDIT_IDS = SMOKE_IDS
audit_q = encode_texts([QUERY_TEXT[q] for q in AUDIT_IDS])

_, pq_ids = search(pq_cpu, audit_q, HIGH_NPROBE)
_, sq64_ids = search(sq_cpu, audit_q, HIGH_NPROBE)
_, sq2_ids = search(sq_cpu, audit_q, MATCHED_LOW_NPROBE)

pq_nd = metrics_from_rows(AUDIT_IDS, pq_ids)[0]
sq64_nd = metrics_from_rows(AUDIT_IDS, sq64_ids)[0]
sq2_nd = metrics_from_rows(AUDIT_IDS, sq2_ids)[0]

audit = {
    "n": len(AUDIT_IDS),
    "pq64_ndcg10": float(pq_nd.mean()),
    "sq64_ndcg10": float(sq64_nd.mean()),
    "sq2_ndcg10": float(sq2_nd.mean()),
    "representation_gap": float(sq64_nd.mean() - pq_nd.mean()),
    "search_gap": float(sq64_nd.mean() - sq2_nd.mean()),
    "fraction_queries_any_relevant_in_sq64_top10": float(np.mean(sq64_nd > 0)),
}
print(json.dumps(audit, indent=2))

# This is a semantic-integrity guard, not a severity-retuning rule.
# A completely wrong row mapping generally collapses qrel effectiveness to ~0.
if audit["fraction_queries_any_relevant_in_sq64_top10"] < 0.05:
    raise RuntimeError(
        "ROW-ID SEMANTIC AUDIT FAILED: BEIR corpus row order likely does not match the frozen FAISS indexes. "
        "Do not run the agent experiment until the original row->doc_id mapping is recovered."
    )

print("ROW-ID SEMANTIC AUDIT — PASS")

## Frozen agent prompt

The agent is deliberately constrained:

- it does **not** answer the question;
- it does **not** expose internal reasoning;
- it emits **one next search query only**;
- it sees evidence text but **not ANN scores**;
- it receives no trajectory history beyond the current query.

This makes the agent a controlled generative state-transition operator rather than a planner benchmark.

In [ ]:
# Cell 6 — Freeze model revision, prompt, sample membership, and runtime protocol BEFORE agent outcomes
SYSTEM_PROMPT = (
    "You are a retrieval query reformulation agent. "
    "Given an original information need, the current retrieval query, and ranked retrieved passages, "
    "produce exactly one concise next search query targeting information still needed to answer the original question. "
    "Do not answer the question. Do not provide explanations or reasoning. "
    "Return only the rewritten search query, with no label, quotation marks, or extra text."
)

def render_prompt(original_question, current_query, evidence):
    parts = [
        "ORIGINAL QUESTION:\n" + original_question.strip(),
        "CURRENT SEARCH QUERY:\n" + current_query.strip(),
        "RETRIEVED PASSAGES:"
    ]
    for rank, item in enumerate(evidence, 1):
        title = (item.get("title") or "").strip()
        text = (item.get("text") or "").replace("\n", " ").strip()
        text = text[:AGENT_MAX_PASSAGE_CHARS]
        parts.append(f"[{rank}] {title}\n{text}")
    parts.append("NEXT SEARCH QUERY:")
    return "\n\n".join(parts)

model_meta = model_info(AGENT_MODEL)
MODEL_REVISION = str(model_meta.sha)

runtime_protocol = {
    "study_id": "ARC-v0.31",
    "status": "FROZEN_BEFORE_GENERATIVE_AGENT_OUTCOMES",
    "dataset": DATASET_ID,
    "encoder": ENCODER_MODEL,
    "agent_backend": AGENT_BACKEND,
    "agent_model": AGENT_MODEL,
    "agent_model_revision": MODEL_REVISION,
    "system_prompt": SYSTEM_PROMPT,
    "generation": {"do_sample": False, "temperature": 0.0, "max_new_tokens": AGENT_MAX_NEW_TOKENS},
    "scores_exposed": False,
    "history_exposed": False,
    "adaptive_stopping": False,
    "H": H,
    "search_k": SEARCH_K,
    "utility_k": UTILITY_K,
    "agent_evidence_k": AGENT_EVIDENCE_K,
    "representation": {"low": str(PQ_PATH), "low_nprobe": 64, "high": str(SQ_PATH), "high_nprobe": 64},
    "search_effort": {"low": str(SQ_PATH), "low_nprobe": 2, "high": str(SQ_PATH), "high_nprobe": 64},
    "v028_endpoint_source": str(V028_ENDPOINT),
    "smoke_ids": SMOKE_IDS,
    "main_ids": MAIN_IDS,
    "seed": SEED,
    "primary": "H3abs_representation_minus_search_effort",
    "primary_direction": "two-sided",
    "bootstrap_reps": BOOTSTRAP_REPS,
    "retention_rule": (
        "Retain positive, null, reversed, or qualitatively different results. "
        "No prompt/model/comparator/horizon/endpoint/main-subset retuning after main outcomes begin."
    ),
}

runtime_text = json.dumps(runtime_protocol, indent=2, sort_keys=True) + "\n"
RUNTIME_SHA = hashlib.sha256(runtime_text.encode()).hexdigest()

PROTOCOL_PATH = OUT_ROOT / "V031_FROZEN_PROTOCOL.json"
if PROTOCOL_PATH.exists():
    old = hashlib.sha256(PROTOCOL_PATH.read_bytes()).hexdigest()
    if old != RUNTIME_SHA:
        raise RuntimeError(
            "A different V031_FROZEN_PROTOCOL.json already exists. "
            "Do not overwrite it after outcomes. Create a new study version if the design must change."
        )
else:
    PROTOCOL_PATH.write_text(runtime_text, encoding="utf-8")
    (OUT_ROOT / "V031_PROTOCOL_SHA256.txt").write_text(
        f"{RUNTIME_SHA}  {PROTOCOL_PATH.name}\n", encoding="utf-8"
    )

print("Agent model:", AGENT_MODEL)
print("Resolved revision:", MODEL_REVISION)
print("Runtime protocol SHA:", RUNTIME_SHA)
print("PROTOCOL FREEZE — PASS")

In [ ]:
# Cell 7 — Load deterministic local LLM agent and persistent response cache
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(AGENT_MODEL, revision=MODEL_REVISION)
agent_model = AutoModelForCausalLM.from_pretrained(
    AGENT_MODEL,
    revision=MODEL_REVISION,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
)
if not torch.cuda.is_available():
    agent_model = agent_model.to("cpu")
agent_model.eval()

CACHE_PATH = OUT_ROOT / "v031_agent_response_cache.jsonl"
response_cache = {}
if CACHE_PATH.exists():
    for line in CACHE_PATH.read_text(encoding="utf-8").splitlines():
        if line.strip():
            obj = json.loads(line)
            response_cache[obj["key"]] = obj["output"]

def cache_key(system, prompt):
    payload = {
        "model": AGENT_MODEL,
        "revision": MODEL_REVISION,
        "system": system,
        "prompt": prompt,
        "do_sample": False,
        "max_new_tokens": AGENT_MAX_NEW_TOKENS,
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()

def clean_query(text):
    text = text.strip()
    text = re.sub(r"^```(?:text)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    text = re.sub(r"^(next search query|query)\s*:\s*", "", text, flags=re.I)
    text = text.strip().strip('"').strip("'").strip()
    text = " ".join(text.split())
    return text[:512]

@torch.inference_mode()
def agent_rewrite(original_question, current_query, evidence, use_cache=True, persist_cache=True):
    user_prompt = render_prompt(original_question, current_query, evidence)
    key = cache_key(SYSTEM_PROMPT, user_prompt)
    if use_cache and key in response_cache:
        return response_cache[key]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=4096)
    device = next(agent_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    out = agent_model.generate(
        **inputs,
        do_sample=False,
        max_new_tokens=AGENT_MAX_NEW_TOKENS,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    result = clean_query(tokenizer.decode(new_tokens, skip_special_tokens=True))
    if not result:
        result = current_query

    if persist_cache:
        response_cache[key] = result
        with CACHE_PATH.open("a", encoding="utf-8") as f:
            f.write(json.dumps({"key": key, "output": result}, ensure_ascii=False) + "\n")
    return result

print("Cached generations:", len(response_cache))
print("AGENT LOAD — PASS")

In [ ]:
# Cell 8 — Evidence accessor and same-evidence determinism audit
def docs_for_rows(rows, k=AGENT_EVIDENCE_K):
    out = []
    for r in rows[:k]:
        r = int(r)
        if r < 0:
            continue
        doc_id = str(ROW_DOCIDS[r])
        d = DOC_STORE.get(doc_id)
        out.append({
            "doc_id": doc_id,
            "title": getattr(d, "title", "") or "",
            "text": getattr(d, "text", "") or "",
        })
    return out

# Determinism audit on 4 smoke queries using the exact same evidence/prompt.
repeat_rows = []
for qid, rows in zip(SMOKE_IDS[:4], sq64_ids[:4]):
    evidence = docs_for_rows(rows)
    q0 = QUERY_TEXT[qid]
    # True raw same-evidence repeat audit: bypass both cache reads and cache writes.
    outputs = [
        agent_rewrite(q0, q0, evidence, use_cache=False, persist_cache=False)
        for _ in range(3)
    ]
    repeat_rows.append({
        "query_id": qid,
        "out1": outputs[0],
        "out2": outputs[1],
        "out3": outputs[2],
        "all_equal": len(set(outputs)) == 1,
    })

repeat_df = pd.DataFrame(repeat_rows)
display(repeat_df[["query_id", "all_equal", "out1"]])
if not repeat_df["all_equal"].all():
    raise RuntimeError(
        "Same-evidence repeat audit is not deterministic. "
        "Do not interpret branch divergence until agent stochasticity is explicitly modeled."
    )
print("SAME-EVIDENCE DETERMINISM AUDIT — PASS")

### Important

The repeat audit above **bypasses the persistent cache** and repeats the identical prompt three times on an engineering-only subset. With deterministic local decoding it should be exactly stable; a future provider-backed API can retain this same audit to estimate raw model nondeterminism.

In [ ]:
# Cell 9 — Coupled generative-agent trajectory runner with checkpointing

RUN_ROOT = OUT_ROOT / "runs"
RUN_ROOT.mkdir(exist_ok=True)
CHECKPOINT_DIR = RUN_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

def lexical_jaccard(a, b):
    A = set(re.findall(r"\w+", a.lower()))
    B = set(re.findall(r"\w+", b.lower()))
    if not A and not B:
        return 0.0
    return 1.0 - len(A & B) / max(len(A | B), 1)

def candidate_jaccard(a, b):
    A, B = set(map(int, a)), set(map(int, b))
    return 1.0 - len(A & B) / max(len(A | B), 1)

def slope_rows(y):
    y = np.asarray(y, dtype=np.float64)
    x = np.arange(y.shape[1], dtype=np.float64)
    xc = x - x.mean()
    return (y @ xc) / np.dot(xc, xc)

def get_pair(mechanism):
    if mechanism == "representation":
        return (pq_cpu, HIGH_NPROBE), (sq_cpu, HIGH_NPROBE)
    if mechanism == "search_effort":
        return (sq_cpu, MATCHED_LOW_NPROBE), (sq_cpu, HIGH_NPROBE)
    raise ValueError(mechanism)

def run_agent_mechanism(qids, mechanism, tag):
    cp = CHECKPOINT_DIR / f"{tag}__{mechanism}.parquet"
    if cp.exists():
        print("skip existing:", cp)
        return pd.read_parquet(cp)

    (idxL, npL), (idxH, npH) = get_pair(mechanism)

    state = {qid: {"L": QUERY_TEXT[qid], "H": QUERY_TEXT[qid]} for qid in qids}
    records = []

    for t in range(H + 1):
        curL = [state[q]["L"] for q in qids]
        curH = [state[q]["H"] for q in qids]

        embL = encode_texts(curL, batch_size=128)
        embH = encode_texts(curH, batch_size=128)

        sL, iL = search(idxL, embL, npL)
        sH, iH = search(idxH, embH, npH)

        ndL, mrL, rcL = metrics_from_rows(qids, iL)
        ndH, mrH, rcH = metrics_from_rows(qids, iH)

        sem_div = 1.0 - np.sum(embL * embH, axis=1)

        for j, qid in enumerate(qids):
            records.append({
                "query_id": qid,
                "mechanism": mechanism,
                "t": t,
                "query_L": curL[j],
                "query_H": curH[j],
                "semantic_query_divergence": float(sem_div[j]),
                "lexical_query_divergence": float(lexical_jaccard(curL[j], curH[j])),
                "candidate_jaccard_divergence": float(candidate_jaccard(iL[j], iH[j])),
                "ndcg10_L": float(ndL[j]),
                "ndcg10_H": float(ndH[j]),
                "mrr10_L": float(mrL[j]),
                "mrr10_H": float(mrH[j]),
                "recall10_L": float(rcL[j]),
                "recall10_H": float(rcH[j]),
                "ndcg10_abs_gap": float(abs(ndH[j] - ndL[j])),
                "ndcg10_signed_gap": float(ndH[j] - ndL[j]),
                "rows_L": json.dumps(list(map(int, iL[j][:UTILITY_K]))),
                "rows_H": json.dumps(list(map(int, iH[j][:UTILITY_K]))),
            })

        if t == H:
            break

        nextL, nextH = [], []
        for j, qid in enumerate(tqdm(qids, desc=f"{tag} {mechanism} rewrite t={t}")):
            original = QUERY_TEXT[qid]
            evL = docs_for_rows(iL[j])
            evH = docs_for_rows(iH[j])
            nextL.append(agent_rewrite(original, curL[j], evL))
            nextH.append(agent_rewrite(original, curH[j], evH))

        for qid, ql, qh in zip(qids, nextL, nextH):
            state[qid]["L"] = ql
            state[qid]["H"] = qh

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    df = pd.DataFrame(records)
    df.to_parquet(cp, index=False)
    print("wrote:", cp, "rows:", len(df))
    return df

In [ ]:
# Cell 10 — Engineering-only 32-query smoke run
smoke_rep = run_agent_mechanism(SMOKE_IDS, "representation", "smoke")
smoke_sea = run_agent_mechanism(SMOKE_IDS, "search_effort", "smoke")

def summarize_smoke(df):
    return {
        "rows": len(df),
        "queries": df.query_id.nunique(),
        "empty_L": int((df.query_L.str.len() == 0).sum()),
        "empty_H": int((df.query_H.str.len() == 0).sum()),
        "mean_semantic_div_final": float(df[df.t == H].semantic_query_divergence.mean()),
        "mean_ndcg_gap_final": float(df[df.t == H].ndcg10_abs_gap.mean()),
    }

print("Representation smoke:", summarize_smoke(smoke_rep))
print("Search smoke:", summarize_smoke(smoke_sea))

# Engineering-only validation: no empty queries and exact expected round count.
for name, df in [("representation", smoke_rep), ("search_effort", smoke_sea)]:
    assert df.query_id.nunique() == N_SMOKE
    assert len(df) == N_SMOKE * (H + 1)
    assert (df.query_L.str.len() > 0).all()
    assert (df.query_H.str.len() > 0).all()

print("ENGINEERING SMOKE — PASS")
print("Do not tune scientific settings based on the smoke effect direction.")

## Main validation gate

**After the next cell starts, the 500-query agent outcomes are the frozen main audit.**

Allowed before starting:
- fix a crash,
- fix a row-mapping error,
- fix an empty-output parsing bug.

Not allowed:
- change the prompt because results "look weak",
- change the LLM,
- change nprobe,
- change H,
- replace the primary endpoint,
- choose a different query subset,
- expose ANN scores to the model.

Any substantive design change requires **ARC-v0.31.1** with a new protocol.

In [ ]:
# Cell 11 — Frozen 500-query main generative-agent run
main_rep = run_agent_mechanism(MAIN_IDS, "representation", "main")
main_sea = run_agent_mechanism(MAIN_IDS, "search_effort", "main")

assert len(main_rep) == N_MAIN * (H + 1)
assert len(main_sea) == N_MAIN * (H + 1)

print("MAIN GENERATIVE-AGENT TRAJECTORIES — COMPLETE")

In [ ]:
# Cell 12 — Convert trajectories to per-query endpoints
def per_query_endpoints(df):
    rows = []
    for qid, z in df.groupby("query_id", sort=False):
        z = z.sort_values("t")
        assert z.t.tolist() == list(range(H + 1))
        arr_abs = z.ndcg10_abs_gap.to_numpy()
        arr_signed = z.ndcg10_signed_gap.to_numpy()
        arr_sem = z.semantic_query_divergence.to_numpy()
        arr_cand = z.candidate_jaccard_divergence.to_numpy()
        arr_mrr = np.abs(z.mrr10_H.to_numpy() - z.mrr10_L.to_numpy())
        arr_rec = np.abs(z.recall10_H.to_numpy() - z.recall10_L.to_numpy())

        def sl(a):
            x = np.arange(len(a), dtype=float)
            xc = x - x.mean()
            return float(a @ xc / (xc @ xc))

        rows.append({
            "query_id": qid,
            "mechanism": z.mechanism.iloc[0],
            "A1_semantic_query_slope": sl(arr_sem),
            "A2_candidate_jaccard_slope": sl(arr_cand),
            "H3_abs_slope": sl(arr_abs),
            "H3_signed_slope": sl(arr_signed),
            "MRR_H3_abs_slope": sl(arr_mrr),
            "Recall_H3_abs_slope": sl(arr_rec),
            "abs_gap_final": float(arr_abs[-1]),
            "abs_gap_final_minus_initial": float(arr_abs[-1] - arr_abs[0]),
            "semantic_query_final": float(arr_sem[-1]),
            "candidate_jaccard_final": float(arr_cand[-1]),
            "higher_fidelity_ahead_final": bool(arr_signed[-1] > 0),
        })
    return pd.DataFrame(rows)

ep_rep = per_query_endpoints(main_rep)
ep_sea = per_query_endpoints(main_sea)
endpoints = pd.concat([ep_rep, ep_sea], ignore_index=True)
ENDPOINT_PATH = RUN_ROOT / "v031_agent_endpoints.parquet"
endpoints.to_parquet(ENDPOINT_PATH, index=False)
display(endpoints.groupby("mechanism").mean(numeric_only=True))
print("ENDPOINTS:", ENDPOINT_PATH)

In [ ]:
# Cell 13 — Frozen paired-query bootstrap primary + secondary contrasts
def paired_bootstrap(x, seed, reps=BOOTSTRAP_REPS):
    x = np.asarray(x, dtype=np.float64)
    assert np.isfinite(x).all()
    n = len(x)
    rg = np.random.default_rng(seed)
    boots = np.empty(reps, dtype=np.float64)
    chunk = 250
    pos = 0
    while pos < reps:
        b = min(chunk, reps - pos)
        idx = rg.integers(0, n, size=(b, n), dtype=np.int32)
        boots[pos:pos+b] = x[idx].mean(axis=1)
        pos += b
    return {
        "mean": float(x.mean()),
        "ci95_low": float(np.quantile(boots, .025)),
        "ci95_high": float(np.quantile(boots, .975)),
        "n_queries": int(n),
    }

rep = ep_rep.set_index("query_id").sort_index()
sea = ep_sea.set_index("query_id").sort_index()
common = rep.index.intersection(sea.index)
assert len(common) == N_MAIN

measures = [
    "H3_abs_slope",
    "A1_semantic_query_slope",
    "A2_candidate_jaccard_slope",
    "H3_signed_slope",
    "MRR_H3_abs_slope",
    "Recall_H3_abs_slope",
    "abs_gap_final",
    "abs_gap_final_minus_initial",
    "semantic_query_final",
    "candidate_jaccard_final",
]

rows = []
for j, measure in enumerate(measures):
    diff = (rep.loc[common, measure].astype(float) - sea.loc[common, measure].astype(float)).to_numpy()
    st = paired_bootstrap(diff, SEED + 31000 + j)
    rows.append({
        "measure": measure,
        "estimand": "representation_minus_search_effort",
        **st,
    })

summary = pd.DataFrame(rows)
SUMMARY_PATH = RUN_ROOT / "v031_agent_paired_query_bootstrap.csv"
summary.to_csv(SUMMARY_PATH, index=False)
display(summary)

primary = summary[summary.measure.eq("H3_abs_slope")].iloc[0]
if primary.ci95_low > 0:
    outcome = "POSITIVE_TRANSFER"
elif primary.ci95_high < 0:
    outcome = "NEGATIVE_TRANSFER_OR_REVERSAL"
else:
    outcome = "UNRESOLVED"

print("\nPRIMARY:", primary.to_dict())
print("CLASSIFICATION:", outcome)

In [ ]:
# Cell 14 — Agent-specific descriptive diagnostics and final gate
# Branch query identity/disagreement at each round.
diag = []
for mechanism, df in [("representation", main_rep), ("search_effort", main_sea)]:
    for t, z in df.groupby("t"):
        diag.append({
            "mechanism": mechanism,
            "t": int(t),
            "mean_semantic_query_divergence": float(z.semantic_query_divergence.mean()),
            "median_semantic_query_divergence": float(z.semantic_query_divergence.median()),
            "mean_candidate_jaccard_divergence": float(z.candidate_jaccard_divergence.mean()),
            "mean_abs_ndcg_gap": float(z.ndcg10_abs_gap.mean()),
            "fraction_exact_same_query_string": float((z.query_L == z.query_H).mean()),
            "fraction_higher_fidelity_ahead": float((z.ndcg10_signed_gap > 0).mean()),
        })

diag_df = pd.DataFrame(diag)
DIAG_PATH = RUN_ROOT / "v031_agent_round_diagnostics.csv"
diag_df.to_csv(DIAG_PATH, index=False)
display(diag_df)

gate = {
    "study_id": "ARC-v0.31",
    "status": "MAIN_GENERATIVE_AGENT_ANALYZED",
    "protocol_sha256": RUNTIME_SHA,
    "agent_model": AGENT_MODEL,
    "agent_model_revision": MODEL_REVISION,
    "n_main_queries": N_MAIN,
    "H": H,
    "primary": {
        "estimand": "representation_minus_search_effort_H3_abs_slope",
        "mean": float(primary["mean"]),
        "ci95": [float(primary["ci95_low"]), float(primary["ci95_high"])],
        "classification": outcome,
        "direction": "two-sided",
    },
    "retuning_performed_after_main_started": False,
    "interpretation": {
        "POSITIVE_TRANSFER": "centroid-era representation-over-search H3 ordering transfers to the frozen generative rewrite operator",
        "NEGATIVE_TRANSFER_OR_REVERSAL": "the generative operator reverses the mechanism ordering",
        "UNRESOLVED": "the mechanism ordering does not resolve under this generative operator/sample",
    }[outcome],
}

GATE_PATH = RUN_ROOT / "v031_agent_primary_gate.json"
GATE_PATH.write_text(json.dumps(gate, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps(gate, indent=2))

In [ ]:
# Cell 15 — Artifact SHA manifest
def sha256_file(path, chunk=8*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

artifact_paths = [
    PROTOCOL_PATH,
    OUT_ROOT / "V031_PROTOCOL_SHA256.txt",
    CACHE_PATH,
    CHECKPOINT_DIR / "main__representation.parquet",
    CHECKPOINT_DIR / "main__search_effort.parquet",
    ENDPOINT_PATH,
    SUMMARY_PATH,
    DIAG_PATH,
    GATE_PATH,
]
manifest = []
for p in artifact_paths:
    if Path(p).exists():
        manifest.append({
            "file": str(p),
            "bytes": Path(p).stat().st_size,
            "sha256": sha256_file(p),
        })

manifest_df = pd.DataFrame(manifest)
MANIFEST_PATH = RUN_ROOT / "V031_ARTIFACT_SHA256.csv"
manifest_df.to_csv(MANIFEST_PATH, index=False)
display(manifest_df)

print("="*92)
print("ARC-v0.31 COMPLETE")
print("PRIMARY CLASSIFICATION:", outcome)
print("OUT:", RUN_ROOT)
print("="*92)

## What to send back

After the **main** run finishes, send either the executed notebook or these files:

1. `v031_agent_primary_gate.json`
2. `v031_agent_paired_query_bootstrap.csv`
3. `v031_agent_round_diagnostics.csv`
4. `v031_agent_endpoints.parquet`
5. `V031_FROZEN_PROTOCOL.json`
6. `V031_ARTIFACT_SHA256.csv`

### Paper-facing interpretation

- **Positive transfer:** the existing representation-over-search mechanism ordering survives a generative rewrite operator.
- **Unresolved:** the boundary is operator-specific; this is scientifically useful and must not be tuned away.
- **Negative/reversed:** the agent operator produces a new mechanism boundary, strengthening the paper's operator-dependence thesis.

Do not call a query-string difference “agent harm.” Query divergence, evidence divergence, retrieval utility, stopping behavior, and answer correctness remain distinct estimands.